# Batch update: Vendor Contacts from CSV (Snowflake Notebook)

This notebook performs an **additive merge** on the **Vendor Contacts** column of **EDLDB.SC_SANDBOX.VC_CPFR_VENDOR_EMAIL**, for rows whose **Vendor Number** appears in your upload file.

**Merge behaviour:**
- Existing contacts already in the DB are **preserved**.
- New contacts from the CSV are **appended** (semicolon-delimited).
- Duplicate email addresses are detected **case-insensitively** — `User@Example.com` and `user@example.com` are treated as the same address and will not be doubled.
- The original casing of the first-seen variant is kept in the final string.

**Who this is for:** Operators who are comfortable running notebook cells step-by-step. You do not need to write SQL by hand; the notebook builds it for you.

**Before you run:**
1. Place **Unapplied_Vendors.csv** under the workspace folder **cpfr_uploads** (same pattern as other CPFR batch tools).
2. The CSV must contain columns named exactly **Vendor Number** and **Vendor Contacts** (extra columns are ignored). Column order does not matter.
3. Resolve any **duplicate Vendor Numbers** in the file before the update step; the notebook will stop if duplicates exist.
4. Set **CONFIRM_UPDATE = True** in the designated cell only after you have reviewed the *before* preview.

**Safety:** Rows not listed in the CSV are never updated. Columns other than **Vendor Contacts** are never modified.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
"""
Central settings for this run. Change only what your environment requires.

TARGET_FQN:
  Full name of the table we update (database.schema.table).

CSV_PATH:
  In Snowflake Workspaces, the notebook runs with a working directory that
  includes your project files. Uploads typically live under cpfr_uploads/.

KEY_COL / CONTACT_COL:
  These names must match the CSV headers and the Snowflake column names
  (including spaces). We select columns by name so the spreadsheet layout
  can vary as long as these two headers exist.
"""

from pathlib import Path
from datetime import datetime

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

TARGET_DB = "EDLDB"
TARGET_SCHEMA = "SC_SANDBOX"
TARGET_TABLE = "VC_CPFR_VENDOR_EMAIL"
TARGET_FQN = f"{TARGET_DB}.{TARGET_SCHEMA}.{TARGET_TABLE}"

KEY_COL = "Vendor Number"
CONTACT_COL = "Vendor Contacts"

TEMP_STAGE_NAME = f"TMP_VC_CPFR_VENDOR_CONTACTS_UPDATE_{RUN_TS}"

CSV_PATH = Path.cwd() / "Unapplied_Vendors.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH}\n"
        "Upload Unapplied_Vendors.csv to cpfr_uploads in this workspace."
    )

print("Target table     :", TARGET_FQN)
print("Temp stage name  :", TEMP_STAGE_NAME)
print("CSV path         :", CSV_PATH)
print("Key column       :", KEY_COL)
print("Contact column   :", CONTACT_COL)

In [ ]:
# =============================================================================
# SNOWFLAKE SESSION AND SQL HELPERS
# =============================================================================
"""
Snowflake Notebooks already authenticate you. We use Snowpark's
get_active_session() instead of username/password connectors.

Define sf_execute and sf_query_df BEFORE calling them (unlike some older
notebooks that call USE before defs).
"""

import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()


def sf_execute(sql: str) -> None:
    """Run DDL/DML or session SQL (USE, etc.)."""
    session.sql(sql).collect()


def sf_query_df(sql: str) -> pd.DataFrame:
    """Run a SELECT and return results as a pandas DataFrame."""
    return session.sql(sql).to_pandas()


sf_execute("USE ROLE SC_USER")
sf_execute("USE DATABASE EDLDB")
sf_execute("USE SCHEMA SC_SANDBOX")
# Uncomment if your role requires an explicit warehouse for writes:
# sf_execute("USE WAREHOUSE STREAMLIT_XS_WH")

sf_query_df(
    "SELECT CURRENT_USER() AS u, CURRENT_ROLE() AS r, "
    "CURRENT_DATABASE() AS d, CURRENT_SCHEMA() AS s, CURRENT_WAREHOUSE() AS w"
)

In [ ]:
# =============================================================================
# READ CSV (BY COLUMN NAME), NORMALIZE, SUBSET
# =============================================================================
"""
We read all fields as text to avoid pandas guessing numeric types for vendor IDs.

After stripping headers and cell text, we keep only KEY_COL and CONTACT_COL.
Any other columns in the file are ignored on purpose.
"""


def read_csv_as_strings(path: Path) -> pd.DataFrame:
    """Load CSV with every column as string dtype."""
    return pd.read_csv(path, dtype=str, keep_default_na=True)


def normalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out


def strip_string_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    return out


df_csv_raw = read_csv_as_strings(CSV_PATH)
df_csv = strip_string_fields(normalize_headers(df_csv_raw))

missing = [c for c in (KEY_COL, CONTACT_COL) if c not in df_csv.columns]
if missing:
    raise KeyError(
        f"CSV is missing required column(s): {missing}. "
        f"Found columns: {list(df_csv.columns)}"
    )

df_upload = df_csv[[KEY_COL, CONTACT_COL]].copy()
print("Rows loaded (before dropping bad keys):", len(df_upload))
df_csv_raw.head(10)

In [ ]:
# =============================================================================
# VALIDATION: BLANK KEYS, DUPLICATE VENDOR NUMBERS
# =============================================================================
"""
Blank Vendor Number rows cannot be joined safely; we remove them after reporting.

Duplicate Vendor Numbers would make the UPDATE ambiguous (which contact wins?).
We fail fast and show the duplicate rows so you can fix the spreadsheet.
"""


def find_key_duplicates(df: pd.DataFrame, key_col: str) -> pd.DataFrame:
    mask = df.duplicated(subset=[key_col], keep=False)
    return df.loc[mask].sort_values(key_col)


blank_key = df_upload[KEY_COL].isna() | (df_upload[KEY_COL].astype(str).str.strip() == "")
n_blank = int(blank_key.sum())
print("Rows with blank Vendor Number (will be dropped):", n_blank)
if n_blank:
    display(df_upload.loc[blank_key].head(20))

df_upload = df_upload.loc[~blank_key].copy()
df_upload[KEY_COL] = df_upload[KEY_COL].astype(str).str.strip()

df_dupes = find_key_duplicates(df_upload, KEY_COL)
print("Duplicate Vendor Number rows:", len(df_dupes))
if len(df_dupes) > 0:
    display(df_dupes)
    raise ValueError(
        "Duplicate Vendor Number values in CSV. Fix the file so each vendor "
        "appears once, then re-run from the read cell."
    )

print("Final row count for update:", len(df_upload))
df_upload.head(15)

In [ ]:
# =============================================================================
# STAGE: WRITE UPLOAD TO A SESSION TEMP TABLE
# =============================================================================
"""
Snowpark writes the small two-column frame to a temporary table. The UPDATE
statement in a later cell joins this stage to the real target table.

Only Vendor Number and Vendor Contacts exist in the stage; nothing else from
the CSV is loaded into Snowflake here.
"""

df_stage = df_upload.reset_index(drop=True)
sp_stage = session.create_dataframe(df_stage)
sp_stage.write.mode("overwrite").save_as_table(TEMP_STAGE_NAME, table_type="temp")

sf_query_df(f"SELECT COUNT(*) AS stage_rows FROM {TEMP_STAGE_NAME}")
sf_query_df(f'SELECT * FROM {TEMP_STAGE_NAME} LIMIT 20')

In [ ]:
sql_before = f'''
SELECT
  TRIM(t."{KEY_COL}") AS "{KEY_COL}",
  t."{CONTACT_COL}" AS contacts_in_db,
  s."{CONTACT_COL}" AS contacts_in_csv
FROM {TARGET_FQN} t
INNER JOIN {TEMP_STAGE_NAME} s
  ON TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
ORDER BY 1
'''

df_before = sf_query_df(sql_before)
matched_cnt = len(df_before)
print("Target rows matched by CSV keys (will receive MERGE):", matched_cnt)
display(df_before.head(30))

sql_not_in_target = f'''
SELECT DISTINCT TRIM(s."{KEY_COL}") AS "{KEY_COL}"
FROM {TEMP_STAGE_NAME} s
WHERE NOT EXISTS (
  SELECT 1 FROM {TARGET_FQN} t
  WHERE TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
)
ORDER BY 1
'''

df_missing = sf_query_df(sql_not_in_target)
print("CSV vendor numbers with NO row in target (UPDATE skips these):", len(df_missing))
if len(df_missing) > 0:
    display(df_missing.head(50))

## Confirm before write

Review **contacts_before** and counts above. Then run the next cell and set **`CONFIRM_UPDATE = True`** before running the UPDATE cell.

In [ ]:
# Set to True only after reviewing the before-preview above.
CONFIRM_UPDATE = True

In [ ]:
if not CONFIRM_UPDATE:
    raise RuntimeError(
        "Set CONFIRM_UPDATE = True in the previous cell after you review "
        "the before-preview, then re-run this cell."
    )

update_sql = f"""
UPDATE {TARGET_FQN} t
SET t."{CONTACT_COL}" = merged.merged_contacts
FROM (
    WITH csv_keys AS (
        SELECT DISTINCT TRIM(s."{KEY_COL}") AS vn
        FROM {TEMP_STAGE_NAME} s
    ),
    existing AS (
        SELECT
            TRIM(t."{KEY_COL}")        AS vn,
            TRIM(e.VALUE::STRING)       AS email,
            ROW_NUMBER() OVER (
                PARTITION BY TRIM(t."{KEY_COL}"), LOWER(TRIM(e.VALUE::STRING))
                ORDER BY 1
            ) AS rn
        FROM {TARGET_FQN} t
        INNER JOIN csv_keys k ON TRIM(t."{KEY_COL}") = k.vn,
             LATERAL SPLIT_TO_TABLE(COALESCE(t."{CONTACT_COL}", ''), ';') e
        WHERE TRIM(e.VALUE::STRING) <> ''
    ),
    csv_new AS (
        SELECT
            TRIM(s."{KEY_COL}")        AS vn,
            TRIM(c.VALUE::STRING)       AS email
        FROM {TEMP_STAGE_NAME} s,
             LATERAL SPLIT_TO_TABLE(COALESCE(s."{CONTACT_COL}", ''), ';') c
        WHERE TRIM(c.VALUE::STRING) <> ''
    ),
    combined AS (
        SELECT vn, email, 1 AS src_order FROM existing WHERE rn = 1
        UNION ALL
        SELECT vn, email, 2 AS src_order FROM csv_new
    ),
    deduped AS (
        SELECT
            vn,
            email,
            ROW_NUMBER() OVER (
                PARTITION BY vn, LOWER(email)
                ORDER BY src_order
            ) AS rn
        FROM combined
    )
    SELECT
        vn,
        LISTAGG(email, ';') WITHIN GROUP (ORDER BY email) AS merged_contacts
    FROM deduped
    WHERE rn = 1
    GROUP BY vn
) merged
WHERE TRIM(t."{KEY_COL}") = merged.vn
"""

result_df = sf_query_df(update_sql)
rows_updated = result_df.iloc[0, 0]
print("Rows updated (Snowflake-reported):", rows_updated)

In [ ]:
sql_after = f'''
SELECT
  TRIM(t."{KEY_COL}") AS "{KEY_COL}",
  t."{CONTACT_COL}" AS contacts_after
FROM {TARGET_FQN} t
INNER JOIN {TEMP_STAGE_NAME} s
  ON TRIM(t."{KEY_COL}") = TRIM(s."{KEY_COL}")
ORDER BY 1
'''

df_after = sf_query_df(sql_after)
display(df_after.head(30))

if matched_cnt > 0:
    before_cols = [c for c in df_before.columns if c.upper() in (KEY_COL.upper(), "CONTACTS_IN_DB", "CONTACTS_IN_CSV")]
    compare = df_before[before_cols].merge(
        df_after, on=KEY_COL, how="outer"
    )
    print("Side-by-side (first 25 rows):")
    display(compare.head(25))
else:
    print("No matched rows earlier; nothing to compare.")

---
## Optional: Full-table contact hygiene pass

The cell below normalises **every row** in the target table — deduplicating email addresses case-insensitively and alphabetising the semicolon-delimited list. It is completely independent of the CSV merge above and can be run on its own whenever you want a clean-up.

Set **RUN_HYGIENE = True** before executing.

In [ ]:
RUN_HYGIENE = False

In [ ]:
if not RUN_HYGIENE:
    raise RuntimeError("Set RUN_HYGIENE = True in the previous cell, then re-run this cell.")

hygiene_preview_sql = f'''
WITH rebuilt AS (
    SELECT
        TRIM(t."{KEY_COL}") AS vn,
        LISTAGG(sub.email, ';') WITHIN GROUP (ORDER BY sub.email) AS clean_contacts
    FROM {TARGET_FQN} t,
         LATERAL SPLIT_TO_TABLE(COALESCE(t."{CONTACT_COL}", ''), ';') e,
         LATERAL (
             SELECT TRIM(e.VALUE::STRING) AS email
         ) sub
    WHERE sub.email <> ''
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRIM(t."{KEY_COL}"), LOWER(sub.email)
        ORDER BY 1
    ) = 1
    GROUP BY TRIM(t."{KEY_COL}")
)
SELECT
    t."{KEY_COL}",
    t."{CONTACT_COL}" AS before_contacts,
    r.clean_contacts   AS after_contacts
FROM {TARGET_FQN} t
INNER JOIN rebuilt r ON TRIM(t."{KEY_COL}") = r.vn
WHERE t."{CONTACT_COL}" IS DISTINCT FROM r.clean_contacts
ORDER BY 1
'''

df_dirty = sf_query_df(hygiene_preview_sql)
print(f"Rows that would change: {len(df_dirty)}  (out of total table)")
if len(df_dirty) > 0:
    display(df_dirty.head(30))
else:
    print("Table is already clean — nothing to do.")

In [ ]:
if not RUN_HYGIENE:
    raise RuntimeError("Set RUN_HYGIENE = True first.")

if len(df_dirty) == 0:
    print("No dirty rows detected — skipping update.")
else:
    hygiene_sql = f"""
    UPDATE {TARGET_FQN} t
    SET t."{CONTACT_COL}" = rebuilt.clean_contacts
    FROM (
        WITH split_rows AS (
            SELECT
                TRIM(t."{KEY_COL}")        AS vn,
                TRIM(e.VALUE::STRING)       AS email,
                ROW_NUMBER() OVER (
                    PARTITION BY TRIM(t."{KEY_COL}"), LOWER(TRIM(e.VALUE::STRING))
                    ORDER BY 1
                ) AS rn
            FROM {TARGET_FQN} t,
                 LATERAL SPLIT_TO_TABLE(COALESCE(t."{CONTACT_COL}", ''), ';') e
            WHERE TRIM(e.VALUE::STRING) <> ''
        )
        SELECT
            vn,
            LISTAGG(email, ';') WITHIN GROUP (ORDER BY email) AS clean_contacts
        FROM split_rows
        WHERE rn = 1
        GROUP BY vn
    ) rebuilt
    WHERE TRIM(t."{KEY_COL}") = rebuilt.vn
      AND t."{CONTACT_COL}" IS DISTINCT FROM rebuilt.clean_contacts
    """

    result = sf_query_df(hygiene_sql)
    rows_cleaned = result.iloc[0, 0]
    print(f"Hygiene complete — {rows_cleaned} row(s) normalised.")